Phase 1 - Setup

In [1]:
%%bash
pip install -q "qdrant-client[fastembed]>=1.14.2" sentence-transformers rapidfuzz pyarrow pandas tqdm

if [ ! -f /kaggle/working/qdrant_bin/qdrant ]; then
  mkdir -p /kaggle/working/qdrant_bin /kaggle/working/qdrant_storage
  wget -q https://github.com/qdrant/qdrant/releases/latest/download/qdrant-x86_64-unknown-linux-musl.tar.gz -O /tmp/qdrant.tar.gz
  tar -xzf /tmp/qdrant.tar.gz -C /kaggle/working/qdrant_bin
  chmod +x /kaggle/working/qdrant_bin/qdrant
fi

In [2]:
import os, subprocess, time, urllib.request, json

os.makedirs("/kaggle/working/qdrant_storage", exist_ok=True)
os.makedirs("/kaggle/working/artifacts", exist_ok=True)

qdrant_cfg = """
storage:
  storage_path: /kaggle/working/qdrant_storage
service:
  http_port: 6333
  grpc_port: 6334
telemetry_disabled: true
"""
with open("/kaggle/working/qdrant_config.yaml", "w") as f:
    f.write(qdrant_cfg)

qdrant_proc = subprocess.Popen(
    ["/kaggle/working/qdrant_bin/qdrant", "--config-path", "/kaggle/working/qdrant_config.yaml"],
    stdout=open("/tmp/qdrant.log", "w"),
    stderr=subprocess.STDOUT,
)

for _ in range(30):
    try:
        with urllib.request.urlopen("http://localhost:6333/healthz", timeout=1) as r:
            if r.status == 200:
                print("Qdrant server is UP on http://localhost:6333")
                break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Qdrant failed to start. Check /tmp/qdrant.log")

Qdrant server is UP on http://localhost:6333


Phase 2 - Data loading

In [3]:
import glob, re
import numpy as np
import pandas as pd

parquet_candidates = glob.glob("/kaggle/input/**/recipes.parquet", recursive=True)
assert parquet_candidates, "Could not find recipes.parquet"
RECIPES_PATH = parquet_candidates[0]
print("Loading:", RECIPES_PATH)

cols = [
    "RecipeId", "Name", "CookTime", "PrepTime", "TotalTime",
    "RecipeCategory", "Keywords", "RecipeIngredientQuantities",
    "RecipeIngredientParts", "AggregatedRating", "ReviewCount",
    "RecipeInstructions", "Images"
]
df_raw = pd.read_parquet(RECIPES_PATH, columns=cols)
print(f"Raw rows: {len(df_raw):,}")

ISO_RE = re.compile(r"^P(?:(?P<days>\d+)D)?(?:T(?:(?P<hours>\d+)H)?(?:(?P<mins>\d+)M)?(?:(?P<secs>\d+)S)?)?$")

Loading: /kaggle/input/datasets/irkaal/foodcom-recipes-and-reviews/recipes.parquet
Raw rows: 522,517


In [4]:
df_raw[0:5]

,RecipeId,Name,CookTime,PrepTime,TotalTime,RecipeCategory,Keywords,RecipeIngredientQuantities,RecipeIngredientParts,AggregatedRating,ReviewCount,RecipeInstructions,Images
0,38.0,Low-Fat Berry Blue Frozen Dessert,PT24H,PT45M,PT24H45M,Frozen Desserts,"[Dessert, Low Protein, Low Cholesterol, Health...","[4, 1⁄4, 1, 1]","[blueberries, granulated sugar, vanilla yogurt...",4.5,4.0,"[Toss 2 cups berries with sugar., Let stand fo...",[https://img.sndimg.com/food/image/upload/w_55...
1,39.0,Biryani,PT25M,PT4H,PT4H25M,Chicken Breast,"[Chicken Thigh & Leg, Chicken, Poultry, Meat, ...","[1, 4, 2, 2, 8, 1⁄4, 8, 1⁄2, 1, 1, 1⁄4, 1⁄4, 1...","[saffron, milk, hot green chili peppers, onion...",3.0,1.0,[Soak saffron in warm milk for 5 minutes and p...,[https://img.sndimg.com/food/image/upload/w_55...
2,40.0,Best Lemonade,PT5M,PT30M,PT35M,Beverages,"[Low Protein, Low Cholesterol, Healthy, Summer...","[1 1⁄2, 1, None, 1 1⁄2, None, 3⁄4]","[sugar, lemons, rind of, lemon, zest of, fresh...",4.5,10.0,"[Into a 1 quart Jar with tight fitting lid, pu...",[https://img.sndimg.com/food/image/upload/w_55...
3,41.0,Carina's Tofu-Vegetable Kebabs,PT20M,PT24H,PT24H20M,Soy/Tofu,"[Beans, Vegetable, Low Cholesterol, Weeknight,...","[12, 1, 2, 1, 10, 1, 3, 2, 2, 2, 1, 2, 1⁄2, 1⁄...","[extra firm tofu, eggplant, zucchini, mushroom...",4.5,2.0,"[Drain the tofu, carefully squeezing out exces...",[https://img.sndimg.com/food/image/upload/w_55...
4,42.0,Cabbage Soup,PT30M,PT20M,PT50M,Vegetable,"[Low Protein, Vegan, Low Cholesterol, Healthy,...","[46, 4, 1, 2, 1]","[plain tomato juice, cabbage, onion, carrots, ...",4.5,11.0,"[Mix everything together and bring to a boil.,...",[https://img.sndimg.com/food/image/upload/w_55...


In [5]:
def parse_iso_minutes(val) -> int | None:
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return None
    s = str(val).strip()
    m = ISO_RE.match(s)
    if not m:
        return None
    days = int(m.group("days") or 0)
    hours = int(m.group("hours") or 0)
    mins = int(m.group("mins") or 0)
    secs = int(m.group("secs") or 0)
    total = days * 1440 + hours * 60 + mins + (1 if secs >= 30 else 0)
    return total if total > 0 else None

def to_py_list(val) -> list[str]:
    if val is None:
        return []
    if isinstance(val, (list, tuple, np.ndarray)):
        return [str(x).strip() for x in val if x is not None and str(x).strip()]
    return []

In [6]:
df = df_raw.copy()
df["total_minutes"] = df["TotalTime"].apply(parse_iso_minutes)
df["ingredients_raw_parts"] = df["RecipeIngredientParts"].apply(to_py_list)
df["quantities_raw"] = df["RecipeIngredientQuantities"].apply(to_py_list)
df["keywords_list"] = df["Keywords"].apply(to_py_list)
df["steps_list"] = df["RecipeInstructions"].apply(to_py_list)
df["images_list"] = df["Images"].apply(to_py_list)

In [7]:
# Filter valid recipes: 1..480 minutes, >= 2 ingredients, non-empty title & instructions
df = df[
    df["Name"].notna()
    & (df["total_minutes"].between(1, 480))
    & (df["ingredients_raw_parts"].apply(len).between(2, 30))
    & (df["steps_list"].apply(len) >= 1)
].copy()

df["ReviewCount"] = df["ReviewCount"].fillna(0.0)
df["AggregatedRating"] = df["AggregatedRating"].fillna(0.0)

# Prioritise well-reviewed recipes, take 60,000
TARGET_N = 60_000
df = df.sort_values(["ReviewCount", "AggregatedRating", "RecipeId"], ascending=[False, False, True])
df = df.head(TARGET_N).reset_index(drop=True)
print(f"Filtered dataset size: {len(df):,} recipes")

Filtered dataset size: 60,000 recipes


In [8]:
def combine_qty_ing(qtys, parts):
    out = []
    for i, p in enumerate(parts):
        q = qtys[i] if i < len(qtys) and qtys[i] not in ("NA", "nan", "") else ""
        out.append(f"{q} {p}".strip() if q else p)
    return out

df["ingredients_raw"] = [combine_qty_ing(q, p) for q, p in zip(df["quantities_raw"], df["ingredients_raw_parts"])]
df["source_url"] = df["RecipeId"].astype(int).apply(lambda rid: f"https://www.food.com/recipe/{rid}")
df["image_url"] = df["images_list"].apply(lambda xs: xs[0] if xs else "")

Phase 3 - Ingredient Normalisation 

In [9]:
import re
from collections import Counter
from rapidfuzz import process, fuzz

PREP_WORDS = {
    "fresh", "freshly", "chopped", "finely", "roughly", "coarsely", "diced", "minced",
    "sliced", "thinly", "thickly", "grated", "shredded", "crushed", "ground", "melted",
    "softened", "room", "temperature", "cold", "warm", "hot", "boiling", "chilled",
    "frozen", "thawed", "canned", "drained", "rinsed", "packed", "lightly", "firmly",
    "divided", "optional", "to", "taste", "for", "garnish", "serving", "peeled",
    "seeded", "deveined", "trimmed", "halved", "quartered", "cubed", "julienned",
    "beaten", "whisked", "sifted", "unsalted", "salted", "sweetened", "unsweetened",
    "low", "fat", "nonfat", "reduced", "sodium", "organic", "raw", "cooked", "roasted",
    "toasted", "dried", "dry", "lean", "extra", "virgin", "pure", "plain", "whole",
    "large", "medium", "small", "tiny", "jumbo", "ripe", "unripe", "boneless",
    "skinless", "skin", "on", "bone", "in", "uncooked", "prepared", "instant",
}

UNIT_WORDS = {
    "cup", "cups", "tbsp", "tablespoon", "tablespoons", "tsp", "teaspoon", "teaspoons",
    "oz", "ounce", "ounces", "lb", "lbs", "pound", "pounds", "g", "gram", "grams",
    "kg", "ml", "l", "liter", "liters", "pinch", "dash", "clove", "cloves", "can",
    "cans", "jar", "jars", "package", "packages", "pkg", "pkt", "packet", "packets",
    "bag", "bags", "bunch", "bunches", "sprig", "sprigs", "stalk", "stalks", "head",
    "heads", "slice", "slices", "piece", "pieces", "strip", "strips", "drop", "drops",
    "stick", "sticks", "bottle", "bottles", "box", "boxes", "inch", "inches",
}

# Canonical synonym map (critical culinary & allergen-relevant aliases first)
SYNONYMS = {
    # Produce & aromatics
    "baby spinach": "spinach", "spinach leaf": "spinach", "spinach leaves": "spinach",
    "fresh spinach": "spinach", "frozen spinach": "spinach",
    "yellow onion": "onion", "white onion": "onion", "red onion": "onion", "sweet onion": "onion",
    "vidalia onion": "onion", "spanish onion": "onion", "onions": "onion",
    "green onion": "scallion", "green onions": "scallion", "spring onion": "scallion", "scallions": "scallion",
    "garlic clove": "garlic", "garlic cloves": "garlic", "garlic powder": "garlic powder",
    "roma tomato": "tomato", "plum tomato": "tomato", "cherry tomato": "tomato",
    "grape tomato": "tomato", "tomatoes": "tomato", "canned tomato": "tomato",
    "diced tomato": "tomato", "crushed tomato": "tomato",
    "bell pepper": "bell pepper", "red bell pepper": "bell pepper", "green bell pepper": "bell pepper",
    "yellow bell pepper": "bell pepper", "sweet pepper": "bell pepper",
    "russet potato": "potato", "yukon gold potato": "potato", "red potato": "potato", "potatoes": "potato",
    "sweet potatoes": "sweet potato", "yam": "sweet potato",
    "cilantro leaf": "cilantro", "coriander leaf": "cilantro", "fresh cilantro": "cilantro",
    "flat leaf parsley": "parsley", "italian parsley": "parsley",
    "lemon juice": "lemon juice", "fresh lemon juice": "lemon juice",
    "lime juice": "lime juice", "fresh lime juice": "lime juice",
    # Grains, flours & gluten sources
    "all purpose flour": "wheat flour", "all-purpose flour": "wheat flour", "plain flour": "wheat flour",
    "white flour": "wheat flour", "unbleached flour": "wheat flour", "bread flour": "wheat flour",
    "cake flour": "wheat flour", "pastry flour": "wheat flour", "self rising flour": "wheat flour",
    "self-rising flour": "wheat flour", "whole wheat flour": "whole wheat flour", "flour": "wheat flour",
    "cornstarch": "cornstarch", "corn starch": "cornstarch", "cornflour": "cornstarch",
    "cornmeal": "cornmeal", "polenta": "cornmeal",
    "almond flour": "almond flour", "almond meal": "almond flour",
    "coconut flour": "coconut flour", "rice flour": "rice flour", "tapioca flour": "tapioca starch",
    "oat": "oats", "rolled oat": "oats", "rolled oats": "oats", "old fashioned oats": "oats", "quick oats": "oats",
    "white rice": "rice", "long grain rice": "rice", "basmati rice": "rice", "jasmine rice": "rice",
    "brown rice": "rice", "arborio rice": "rice", "cooked rice": "rice",
    "spaghetti": "pasta", "penne": "pasta", "fettuccine": "pasta", "linguine": "pasta",
    "macaroni": "pasta", "rigatoni": "pasta", "rotini": "pasta", "fusilli": "pasta", "egg noodle": "egg noodles",
    "bread crumb": "breadcrumbs", "bread crumbs": "breadcrumbs", "panko": "breadcrumbs", "panko breadcrumbs": "breadcrumbs",
    # Pantry & sauces
    "extra virgin olive oil": "olive oil", "virgin olive oil": "olive oil",
    "canola oil": "vegetable oil", "sunflower oil": "vegetable oil", "corn oil": "vegetable oil",
    "kosher salt": "salt", "sea salt": "salt", "table salt": "salt",
    "black pepper": "black pepper", "ground black pepper": "black pepper", "pepper": "black pepper",
    "granulated sugar": "sugar", "white sugar": "sugar", "caster sugar": "sugar",
    "light brown sugar": "brown sugar", "dark brown sugar": "brown sugar",
    "powdered sugar": "powdered sugar", "confectioners sugar": "powdered sugar", "icing sugar": "powdered sugar",
    "tamari": "tamari", "gluten free soy sauce": "tamari", "gluten-free soy sauce": "tamari",
    "light soy sauce": "soy sauce", "dark soy sauce": "soy sauce", "low sodium soy sauce": "soy sauce",
    "chicken broth": "chicken stock", "chicken stock": "chicken stock",
    "beef broth": "beef stock", "beef stock": "beef stock",
    "vegetable broth": "vegetable stock", "vegetable stock": "vegetable stock",
    # Dairy & eggs & vegan alternatives
    "eggs": "egg", "large egg": "egg", "egg yolk": "egg", "egg white": "egg", "egg whites": "egg",
    "whole milk": "milk", "skim milk": "milk", "2% milk": "milk", "buttermilk": "buttermilk",
    "heavy cream": "heavy cream", "whipping cream": "heavy cream", "heavy whipping cream": "heavy cream",
    "half and half": "cream", "half-and-half": "cream", "sour cream": "sour cream",
    "cream cheese": "cream cheese", "cheddar cheese": "cheddar", "sharp cheddar": "cheddar",
    "parmesan cheese": "parmesan", "parmigiano reggiano": "parmesan", "mozzarella cheese": "mozzarella",
    "feta cheese": "feta", "goat cheese": "goat cheese", "ricotta cheese": "ricotta",
    "unsalted butter": "butter", "salted butter": "butter",
    "almond milk": "almond milk", "soy milk": "soy milk", "oat milk": "oat milk", "coconut milk": "coconut milk",
    # Proteins
    "chicken breast": "chicken", "chicken breasts": "chicken", "chicken thigh": "chicken",
    "chicken thighs": "chicken", "chicken tender": "chicken", "rotisserie chicken": "chicken",
    "ground beef": "beef", "beef chuck": "beef", "steak": "beef", "stew beef": "beef",
    "pork chop": "pork", "pork tenderloin": "pork", "ground pork": "pork",
    "turkey breast": "turkey", "ground turkey": "turkey",
    "garbanzo bean": "chickpeas", "garbanzo beans": "chickpeas", "chickpea": "chickpeas",
    "black beans": "black bean", "kidney beans": "kidney bean", "cannellini beans": "white bean",
    "firm tofu": "tofu", "extra firm tofu": "tofu", "silken tofu": "tofu",
    # Nuts & seeds
    "peanut": "peanuts", "peanut butter": "peanut butter", "smooth peanut butter": "peanut butter",
    "almond": "almonds", "sliced almonds": "almonds", "slivered almonds": "almonds",
    "walnut": "walnuts", "pecan": "pecans", "cashew": "cashews", "pistachio": "pistachios",
    "hazelnut": "hazelnuts", "pine nut": "pine nuts", "pine nuts": "pine nuts",
}

In [10]:
def simple_singularise(phrase: str) -> str:
    words = phrase.split()
    if not words:
        return phrase
    last = words[-1]
    # Keep common plural canonicals intact
    keep = {"oats", "chickpeas", "lentils", "almonds", "walnuts", "pecans", "cashews",
            "peanuts", "pistachios", "hazelnuts", "breadcrumbs", "noodles", "egg noodles",
            "berries", "molasses", "couscous", "hummus", "asparagus"}
    if phrase in keep or last in keep:
        return phrase
    if len(last) > 4 and last.endswith("ies"):
        words[-1] = last[:-3] + "y"
    elif len(last) > 4 and last.endswith("oes"):
        words[-1] = last[:-2]
    elif len(last) > 3 and last.endswith("s") and not last.endswith(("ss", "us", "is")):
        words[-1] = last[:-1]
    return " ".join(words)

CLEAN_PARENS_RE = re.compile(r"\([^)]*\)")
NON_ALPHA_RE = re.compile(r"[^a-z0-9\s\-]")
QTY_PREFIX_RE = re.compile(r"^[\d\s/.\-]+")

def clean_raw_line(raw: str) -> str:
    s = raw.lower().strip()
    s = CLEAN_PARENS_RE.sub(" ", s)
    s = s.split(",")[0]  # drop ", finely chopped" tail
    s = QTY_PREFIX_RE.sub("", s)
    s = s.replace("-", " ")
    s = NON_ALPHA_RE.sub(" ", s)
    tokens = [t for t in s.split() if t not in UNIT_WORDS and t not in PREP_WORDS and not t.isdigit()]
    cleaned = " ".join(tokens).strip()
    return cleaned

In [11]:
# Build top-500 frequent ingredient vocabulary directly from the 60k recipes
raw_counter = Counter()
for parts in df["ingredients_raw_parts"]:
    for p in parts:
        c = clean_raw_line(p)
        if c:
            c = SYNONYMS.get(c, simple_singularise(c))
            c = SYNONYMS.get(c, c)
            if c:
                raw_counter[c] += 1

TOP_CANONICAL = {ing for ing, _ in raw_counter.most_common(600)}
# Always include all synonym targets + critical safety ingredients
TOP_CANONICAL.update(SYNONYMS.values())
KNOWN_VOCAB = sorted(TOP_CANONICAL)

fuzzy_cache: dict[str, str] = {}
unmatched_counter = Counter()

In [12]:
def normalise_ingredient(raw: str) -> tuple[str, bool]:
    """Returns (canonical_name, is_known)."""
    c = clean_raw_line(raw)
    if not c:
        return ("", False)
    if c in SYNONYMS:
        return (SYNONYMS[c], True)
    cs = simple_singularise(c)
    if cs in SYNONYMS:
        return (SYNONYMS[cs], True)
    if cs in TOP_CANONICAL:
        return (cs, True)
    if c in TOP_CANONICAL:
        return (c, True)
    if cs in fuzzy_cache:
        m = fuzzy_cache[cs]
        return (m, m != "__UNMATCHED__")
    # Conservative high-threshold fuzzy match only for typos (len >= 5)
    if len(cs) >= 5:
        hit = process.extractOne(cs, KNOWN_VOCAB, scorer=fuzz.WRatio, score_cutoff=92)
        if hit:
            fuzzy_cache[cs] = hit[0]
            return (hit[0], True)
    fuzzy_cache[cs] = "__UNMATCHED__"
    unmatched_counter[cs] += 1
    return (cs, False)

canon_lists = []
unknown_flags = []
for parts in df["ingredients_raw_parts"]:
    c_set = []
    has_unknown = False
    for p in parts:
        c, known = normalise_ingredient(p)
        if c:
            if c not in c_set:
                c_set.append(c)
            if not known:
                has_unknown = True
    canon_lists.append(c_set)
    unknown_flags.append(has_unknown)

df["ingredients"] = canon_lists
df["ingredient_count"] = df["ingredients"].apply(len)
df["has_unknown_ingredient"] = unknown_flags

print(f"Unique canonical vocab size: {len(KNOWN_VOCAB)}")
print(f"Recipes with >=1 unrecognised ingredient: {df['has_unknown_ingredient'].sum():,} / {len(df):,}")
print("Top 15 unmatched tokens:", unmatched_counter.most_common(15))

Unique canonical vocab size: 610
Recipes with >=1 unrecognised ingredient: 13,998 / 60,000
Top 15 unmatched tokens: [('cajun seasoning mix', 1), ('round steak', 1), ('hickory smoke salt', 1), ('russet baking potato', 1), ('swiss chard', 1), ('chicken meat', 1), ('beef sirloin', 1), ('dark beer', 1), ('canning salt', 1), ('green seedless grape', 1), ('red seedless grape', 1), ('sweet italian sausage link', 1), ('italian plum tomato', 1), ('mexican style tomato', 1), ('chocolate milk', 1)]
